<a href="https://colab.research.google.com/github/budennovsk/Pandas/blob/master/v6_2_level_model_rerank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install implicit catboost rectools[lightfm]



In [ ]:
from pathlib import Path
import typing as tp
import warnings

import pandas as pd
import numpy as np

from implicit.nearest_neighbours import CosineRecommender
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier
from catboost import CatBoostClassifier, CatBoostRanker
try:
    from lightgbm import LGBMClassifier, LGBMRanker
    LGBM_AVAILABLE = True
except ImportError:
    warnings.warn("lightgbm is not installed. Some parts of the notebook will be skipped.")
    LGBM_AVAILABLE = False
from rectools.dataset import Interactions
from lightfm import LightFM
from rectools import Columns
from rectools.dataset import Dataset
from rectools.metrics import Precision, Recall, MeanInvUserFreq, Serendipity,MAP,NDCG,HitRate,calc_metrics
from rectools.models import (
    ImplicitALSWrapperModel,
    ImplicitBPRWrapperModel,
    LightFMWrapperModel,
    PureSVDModel,
    ImplicitItemKNNWrapperModel,
    EASEModel,
    RandomModel,
    PopularInCategoryModel,
    PopularModel,
    EASEModel, SASRecModel, BERT4RecModel)
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking
from implicit.nearest_neighbours import CosineRecommender
from rectools.models.base import ExternalIds
from rectools.models.ranking import (
    CandidateRankingModel,
    CandidateGenerator,
    Reranker,

    CatBoostReranker,
    CandidateFeatureCollector,
    PerUserNegativeSampler
)
from rectools.models.nn.item_net import CatFeaturesItemNet, IdEmbeddingsItemNet
from rectools.model_selection import cross_validate, TimeRangeSplitter,LastNSplitter,Splitter
from tqdm.auto import tqdm

In [ ]:
path_users = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/users.csv'
path_items = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/items.csv'
path_interactions = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/interactions.csv'


users = pd.read_csv(path_users)
items = pd.read_csv(path_items)
interactions = (
    pd.read_csv(path_interactions, parse_dates=["last_watch_dt"])
    .rename(columns={"last_watch_dt": Columns.Datetime})
)
interactions = interactions.head(1000000)
users_clise = users[users['user_id'].isin(interactions['user_id'].unique())]
items_clise = items[items['item_id'].isin(interactions['item_id'].unique())]
interactions["weight"] = 1
dataset = Dataset.construct(interactions)
RANDOM_STATE = 32


# dataset
# interactions[Columns.Datetime] = pd.to_datetime(interactions[Columns.Datetime], format='%Y-%m-%d')

In [ ]:
interactions users_clise items_clise

In [ ]:
items.info()

In [ ]:
max_date = interactions[Columns.Datetime].max()
train = interactions[interactions[Columns.Datetime] < max_date - pd.Timedelta(days=7)].copy()
test = interactions[interactions[Columns.Datetime] >= max_date - pd.Timedelta(days=7)].copy()

print(f"train: {train.shape}")
print(f"test: {test.shape}")

In [ ]:
# отфильтруем холодных пользователей из теста
cold_users = list(set(test['user_id']) - set(train['user_id']))
len(cold_users)

In [ ]:
import json
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import pandas as pd

from rectools.dataset import Dataset
from rectools.metrics import Precision, Recall, HitRate, NDCG, MAP
from rectools.models import EASEModel, ImplicitALSWrapperModel, LightFMWrapperModel

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TOP_K = 10
CAND_K = 100
K_PER_MODEL = 200
POS_TH = 50.0


@dataclass
class Fold:
    fold_id: int
    train: pd.DataFrame
    val: pd.DataFrame
    test: pd.DataFrame


def make_binary_target(df: pd.DataFrame, th: float = POS_TH) -> pd.DataFrame:
    out = df.copy()
    out["is_pos"] = (out["watched_pct"] >= th).astype("int8")
    return out


def build_folds_expanding(interactions: pd.DataFrame) -> List[Fold]:
    df = interactions.sort_values("datetime").reset_index(drop=True)
    n = len(df)
    qs = [(0.60, 0.80), (0.80, 0.90), (0.90, 0.95)]

    folds = []
    for i, (q_tr, q_val) in enumerate(qs, start=1):
        i_tr = int(n * q_tr)
        i_val = int(n * q_val)
        folds.append(Fold(i, df.iloc[:i_tr].copy(), df.iloc[i_tr:i_val].copy(), df.iloc[i_val:].copy()))
        print(f"Fold {i}: train={i_tr:,} val={i_val-i_tr:,} test={n-i_val:,} "
              f"train_end={df.iloc[:i_tr]['datetime'].max()} val_end={df.iloc[i_tr:i_val]['datetime'].max()} test_end={df.iloc[i_val:]['datetime'].max()}")
    return folds


def drop_fully_cold_users_and_items(fold: Fold) -> Fold:
    train_users = set(fold.train["user_id"].unique())
    train_items = set(fold.train["item_id"].unique())

    def _f(df: pd.DataFrame) -> pd.DataFrame:
        out = df[df["user_id"].isin(train_users)].copy()
        out = out[out["item_id"].isin(train_items)].copy()
        return out

    return Fold(fold.fold_id, _f(fold.train), _f(fold.val), _f(fold.test))


def build_rectools_interactions_canonical(df: pd.DataFrame) -> pd.DataFrame:
    """
    ВАЖНО: делаем ровно 4 колонки с ИМЕНАМИ, которые ожидает rectools в разных версиях:
      user_id, item_id, datetime, weight
    """
    out = pd.DataFrame({
        "user_id": df["user_id"].astype("int64").values,
        "item_id": df["item_id"].astype("int64").values,
        "datetime": pd.to_datetime(df["datetime"], errors="coerce").values,
        "weight": pd.to_numeric(df["watched_pct"], errors="coerce").fillna(0.0).astype("float32").values,
    })
    out = out.dropna(subset=["datetime"]).copy()
    return out


def build_dataset_compat(interactions_rt: pd.DataFrame) -> Dataset:
    """
    Разные версии rectools по-разному принимают аргументы.
    Пробуем несколько вариантов.
    """
    # Вариант 1: interactions_df именованный
    try:
        return Dataset.construct(interactions_df=interactions_rt)
    except Exception:
        pass

    # Вариант 2: просто df позиционно
    try:
        return Dataset.construct(interactions_rt)
    except Exception:
        pass

    # Вариант 3: Dataset(interactions=...) (на всякий)
    try:
        return Dataset(interactions_rt)
    except Exception as e:
        raise RuntimeError(f"Не смог собрать Dataset из rectools. Пример df:\n{interactions_rt.head()}") from e


def rectools_metrics_at_k(k: int = TOP_K):
    return {
        f"Precision@{k}": Precision(k=k),
        f"Recall@{k}": Recall(k=k),
        f"HitRate@{k}": HitRate(k=k),
        f"NDCG@{k}": NDCG(k=k),
        f"MAP@{k}": MAP(k=k),
    }


def eval_metrics(reco: pd.DataFrame, test_pos: pd.DataFrame, k: int = TOP_K) -> Dict[str, float]:
    """
    Для метрик rectools обычно достаточно:
      reco: user_id, item_id, score
      test: user_id, item_id
    """
    reco2 = reco[["user_id", "item_id", "score"]].copy()
    test2 = test_pos[["user_id", "item_id"]].copy()

    out = {}
    for name, m in rectools_metrics_at_k(k).items():
        out[name] = float(m.calc(reco2, test2))
    return out


def fit_models(ds: Dataset) -> Dict[str, object]:
    models = {}

    ease = EASEModel(regularization=500.0)
    ease.fit(ds)
    models["EASE"] = ease

    als = ImplicitALSWrapperModel(
        factors=128, regularization=0.01, iterations=30, alpha=20.0, random_state=RANDOM_SEED
    )
    als.fit(ds)
    models["ALS"] = als

    lfm = LightFMWrapperModel(
        no_components=64, loss="warp", random_state=RANDOM_SEED, epochs=20, num_threads=4
    )
    lfm.fit(ds)
    models["LightFM"] = lfm

    return models


def recommend_union(models: Dict[str, object], ds: Dataset, k_total: int = CAND_K, k_per_model: int = K_PER_MODEL) -> pd.DataFrame:
    recos = []
    for name, model in models.items():
        r = model.recommend(dataset=ds, k=k_per_model, filter_viewed=True)

        # В разных версиях rectools названия колонок могут различаться.
        # Поэтому приводим к user_id/item_id/score через "угадайку".
        cols = list(r.columns)
        # чаще всего это ["user_id", "item_id", "score"]
        if set(["user_id", "item_id"]).issubset(cols):
            rr = r.rename(columns={c: c for c in cols}).copy()
            rr["model"] = name
            rr = rr[["user_id", "item_id", "score", "model"]]
        else:
            # fallback: берём первые 2 колонки как ids + score
            rr = r.copy()
            rr.columns = ["user_id", "item_id", "score"] + list(rr.columns[3:])
            rr["model"] = name
            rr = rr[["user_id", "item_id", "score", "model"]]

        recos.append(rr)

    all_r = pd.concat(recos, ignore_index=True)

    pivot = all_r.pivot_table(
        index=["user_id", "item_id"],
        columns="model",
        values="score",
        aggfunc="max",
        fill_value=0.0,
    ).reset_index()

    pivot = pivot.rename(columns={"EASE": "score_ease", "ALS": "score_als", "LightFM": "score_lightfm"})
    for c in ["score_ease", "score_als", "score_lightfm"]:
        if c not in pivot.columns:
            pivot[c] = 0.0

    pivot["score"] = pivot[["score_ease", "score_als", "score_lightfm"]].sum(axis=1)

    pivot = pivot.sort_values(["user_id", "score"], ascending=[True, False])
    pivot["rank"] = pivot.groupby("user_id").cumcount() + 1
    pivot = pivot[pivot["rank"] <= k_total].drop(columns=["rank"])

    return pivot


def topk(df: pd.DataFrame, k: int = TOP_K) -> pd.DataFrame:
    out = df.sort_values(["user_id", "score"], ascending=[True, False]).copy()
    out["rank"] = out.groupby("user_id").cumcount() + 1
    return out[out["rank"] <= k].drop(columns=["rank"])


def run_candidate_generation_3folds(interactions: pd.DataFrame) -> pd.DataFrame:
    df = interactions.copy()
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df[~df["datetime"].isna()].copy()
    df["watched_pct"] = pd.to_numeric(df["watched_pct"], errors="coerce").fillna(0.0).astype("float32")

    folds = build_folds_expanding(df)
    folds = [drop_fully_cold_users_and_items(f) for f in folds]

    rows = []
    for f in folds:
        print("\n" + "=" * 80)
        print(f"FOLD {f.fold_id}")

        train0 = make_binary_target(f.train)
        test0 = make_binary_target(f.test)

        train_rt = build_rectools_interactions_canonical(train0)
        ds = build_dataset_compat(train_rt)

        models = fit_models(ds)
        candidates = recommend_union(models, ds, k_total=CAND_K, k_per_model=K_PER_MODEL)

        reco_test_top10 = topk(candidates[["user_id", "item_id", "score"]], k=TOP_K)

        test_pos = test0[test0["is_pos"] == 1][["user_id", "item_id"]].drop_duplicates()
        metrics = eval_metrics(reco_test_top10, test_pos, k=TOP_K)

        print("TEST metrics (union candidates):")
        print(json.dumps(metrics, ensure_ascii=False, indent=2))

        row = {"fold": f.fold_id}
        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
results_retrieval = run_candidate_generation_3folds(interactions)
display(results_retrieval)
display(results_retrieval.drop(columns=["fold"]).mean(numeric_only=True))